# A5 — TAD-YOLO
Adds DySample to the T4→T3 and T3→T2 transitions of A4. This is the final TAD-YOLO graph. Use a Kaggle T4 GPU, Internet On, and attach VisDrone.

In [ ]:
from pathlib import Path
import importlib.metadata as md
import subprocess, sys

VARIANT = "a5_tad_yolo"
SEED, EPOCHS, BATCH, IMGSZ, WORKERS = 0, 300, 8, 640, 4
REPO_URL, REPO_REF = "https://github.com/DongTienDinh/TAD-YOLO.git", "kaggle-ablation"
REPO = Path("/kaggle/working/TAD-YOLO")
PROJECT = Path("/kaggle/working/runs/ablation")
RUN_NAME = f"{VARIANT}_seed{SEED}"
RESUME_CHECKPOINT = None

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
required = {"torch": "2.10.0", "torchvision": "0.25.0", "ultralytics-opencv-headless": "8.4.154", "PyYAML": "6.0.3"}
installed = {dist.metadata["Name"].lower(): dist.version for dist in md.distributions()}
missing = [f"{name}=={version}" for name, version in required.items() if installed.get(name.lower(), "").split('+')[0] != version]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", *missing], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO), "--no-deps"], check=True)


In [ ]:
sys.path.insert(0, str(REPO))
import torch, ultralytics
from tad_yolo.offline import resolve_visdrone_dataset, validate_dataset_yaml
assert torch.cuda.is_available() and torch.cuda.device_count() >= 1
assert "T4" in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
assert ultralytics.__version__ == "8.4.154"
matches = list(Path("/kaggle/input").glob("**/visdrone_kaggle.yaml"))
DATASET_YAML = matches[0] if len(matches) == 1 else resolve_visdrone_dataset()
print(torch.cuda.get_device_name(0), torch.__version__, ultralytics.__version__)
print(validate_dataset_yaml(DATASET_YAML))


In [ ]:
base = [sys.executable, str(REPO / "scripts/train_ablation.py"), "--variant", VARIANT, "--device", "0", "--project", str(PROJECT)]
subprocess.run(base + ["--dry-run"], check=True)


In [ ]:
cmd = base + ["--dataset", str(DATASET_YAML), "--epochs", str(EPOCHS), "--batch", str(BATCH), "--imgsz", str(IMGSZ), "--workers", str(WORKERS), "--seed", str(SEED), "--precision", "fp16", "--name", RUN_NAME]
if RESUME_CHECKPOINT:
    cmd += ["--resume", str(RESUME_CHECKPOINT)]
subprocess.run(cmd, check=True)


In [ ]:
import shutil
candidates = [p for p in PROJECT.glob(f"{RUN_NAME}*") if (p / "weights/last.pt").is_file()]
assert candidates, f"No completed run found under {PROJECT}"
run_dir = max(candidates, key=lambda p: (p / "weights/last.pt").stat().st_mtime)
archive = shutil.make_archive(str(Path("/kaggle/working") / RUN_NAME), "zip", run_dir)
print("Best:", run_dir / "weights/best.pt")
print("Archive:", archive)
